In [39]:
!uv pip install requests pandas pyarrow python-dotenv jupyter ipykernel


Using Python 3.11.3 environment at: /Users/jacodon/Dev/data-platform/.venv
Checked 6 packages in 200ms


requests — fa le chiamate HTTP all'API di Alpha Vantage. È la libreria standard per fare richieste web in Python.  
pandas — manipola i dati in formato tabellare (DataFrame). Lo useremo per normalizzare il JSON dell'API e prepararlo per la conversione in Parquet.  
pyarrow — gestisce il formato Parquet. Pandas lo usa internamente per leggere e scrivere file .parquet.  
python-dotenv — carica il .env in memoria come abbiamo discusso.  
jupyter — il core di Jupyter, necessario per eseguire i notebook.  
ipykernel — permette al .venv di essere riconosciuto come kernel da VS Code. Senza di esso VS Code non saprebbe come collegare il notebook al venv.  

In [40]:
import sys # per modificare il path di ricerca dei moduli
import os # per accedere alle variabili d'ambiente e al file system
import requests # per fare richieste HTTP all'API di Alpha Vantage
import pandas as pd # per manipolare i dati in formato tabellare
from dotenv import load_dotenv  # per caricare le variabili d'ambiente da un file .env

sys.path.append('.')   # aggiunge ingestion/ al path di ricerca

load_dotenv(dotenv_path='../.env')  # carica le variabili d'ambiente dal file .env
API_KEY = os.getenv('ALPHA_VANTAGE_API_KEY') # recupera la chiave API dalle variabili d'ambiente
print(API_KEY) # per verificare che la chiave API sia stata caricata correttamente


CPQV7Y207E1YCQNB


In [41]:
ticker = "AAPL" # simbolo del titolo da analizzare, ad esempio Apple

In [42]:
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": ticker,
    "outputsize": "compact",  # ultimi 100 giorni
    "apikey": API_KEY
}

response = requests.get("https://www.alphavantage.co/query", params=params)
print(response.url)

data = response.json()
print(type(data))  # per verificare che la risposta sia un dizionario
print(data.keys())  # per vedere le chiavi principali del dizionario

https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=AAPL&outputsize=compact&apikey=CPQV7Y207E1YCQNB
<class 'dict'>
dict_keys(['Meta Data', 'Time Series (Daily)'])


In [43]:
print(data['Meta Data'])

{'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'AAPL', '3. Last Refreshed': '2026-06-08', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}


In [44]:
print(data['Time Series (Daily)'])

{'2026-06-08': {'1. open': '308.7390', '2. high': '317.4000', '3. low': '301.1700', '4. close': '301.5400', '5. volume': '77949082'}, '2026-06-05': {'1. open': '312.8600', '2. high': '315.1700', '3. low': '307.1500', '4. close': '307.3400', '5. volume': '65310502'}, '2026-06-04': {'1. open': '313.2300', '2. high': '313.5400', '3. low': '309.6500', '4. close': '311.2300', '5. volume': '44869134'}, '2026-06-03': {'1. open': '314.1750', '2. high': '316.9400', '3. low': '308.8500', '4. close': '310.2600', '5. volume': '50836705'}, '2026-06-02': {'1. open': '307.4600', '2. high': '315.4500', '3. low': '306.6850', '4. close': '315.2000', '5. volume': '44534716'}, '2026-06-01': {'1. open': '309.6250', '2. high': '310.9400', '3. low': '305.0200', '4. close': '306.3100', '5. volume': '48849933'}, '2026-05-29': {'1. open': '311.7750', '2. high': '315.0000', '3. low': '309.5300', '4. close': '312.0600', '5. volume': '70026752'}, '2026-05-28': {'1. open': '310.6800', '2. high': '312.8000', '3. low

In [45]:
import json
print(json.dumps(data, indent=2))

{
  "Meta Data": {
    "1. Information": "Daily Prices (open, high, low, close) and Volumes",
    "2. Symbol": "AAPL",
    "3. Last Refreshed": "2026-06-08",
    "4. Output Size": "Compact",
    "5. Time Zone": "US/Eastern"
  },
  "Time Series (Daily)": {
    "2026-06-08": {
      "1. open": "308.7390",
      "2. high": "317.4000",
      "3. low": "301.1700",
      "4. close": "301.5400",
      "5. volume": "77949082"
    },
    "2026-06-05": {
      "1. open": "312.8600",
      "2. high": "315.1700",
      "3. low": "307.1500",
      "4. close": "307.3400",
      "5. volume": "65310502"
    },
    "2026-06-04": {
      "1. open": "313.2300",
      "2. high": "313.5400",
      "3. low": "309.6500",
      "4. close": "311.2300",
      "5. volume": "44869134"
    },
    "2026-06-03": {
      "1. open": "314.1750",
      "2. high": "316.9400",
      "3. low": "308.8500",
      "4. close": "310.2600",
      "5. volume": "50836705"
    },
    "2026-06-02": {
      "1. open": "307.4600",
   

In [46]:
# Facciamo alcuni test per capire meglio la struttura dei dati

time_series = data['Time Series (Daily)'] # è un dizionario dove le chiavi sono le date e i valori sono altri dizionari con i prezzi
print(type(time_series)) # dovrebbe essere un dizionario
print(len(time_series)) # dovrebbe essere 100 per la modalità "compact"

<class 'dict'>
100


In [47]:
# estraiamo il primo elemento per vedere la struttura

first_date = list(time_series.keys())[0] # la prima data dovrebbe essere la più recente, ma è meglio verificarlo
print(f"Data più recente: {first_date}") 
print(time_series[first_date])

Data più recente: 2026-06-08
{'1. open': '308.7390', '2. high': '317.4000', '3. low': '301.1700', '4. close': '301.5400', '5. volume': '77949082'}


In [48]:
df = pd.DataFrame.from_dict(time_series, orient='index') # orient='index' perché le chiavi del dizionario (le date) devono diventare gli indici del DataFrame
print(df.head())
print(df.dtypes)

             1. open   2. high    3. low  4. close 5. volume
2026-06-08  308.7390  317.4000  301.1700  301.5400  77949082
2026-06-05  312.8600  315.1700  307.1500  307.3400  65310502
2026-06-04  313.2300  313.5400  309.6500  311.2300  44869134
2026-06-03  314.1750  316.9400  308.8500  310.2600  50836705
2026-06-02  307.4600  315.4500  306.6850  315.2000  44534716
1. open      str
2. high      str
3. low       str
4. close     str
5. volume    str
dtype: object


In [49]:
# rinomina le colonne
df.columns = ['open', 'high', 'low', 'close', 'volume'] # attributo dell'oggetto DataFrame che permette di rinominare le colonne

# converti i tipi
df[['open', 'high', 'low', 'close']] = df[['open', 'high', 'low', 'close']].astype(float) # metodo astype() per convertire i tipi di dati, in questo caso da stringa a float
df['volume'] = df['volume'].astype(int) # il volume è un numero intero, quindi lo convertiamo in int

# converti l'indice in datetime
df.index = pd.to_datetime(df.index) # metodo to_datetime() per convertire le date in formato datetime
df.index.name = 'date' # rinomina l'indice in 'date' per chiarezza

# aggiungi il ticker
df['ticker'] = ticker # aggiungiamo una colonna 'ticker' con il ticker  del titolo

print(df.head()) 
print(df.dtypes)

               open    high      low   close    volume ticker
date                                                         
2026-06-08  308.739  317.40  301.170  301.54  77949082   AAPL
2026-06-05  312.860  315.17  307.150  307.34  65310502   AAPL
2026-06-04  313.230  313.54  309.650  311.23  44869134   AAPL
2026-06-03  314.175  316.94  308.850  310.26  50836705   AAPL
2026-06-02  307.460  315.45  306.685  315.20  44534716   AAPL
open      float64
high      float64
low       float64
close     float64
volume      int64
ticker        str
dtype: object


In [51]:
df = df.reset_index()
print(df.head())
print(df.dtypes)

        date     open    high      low   close    volume ticker
0 2026-06-08  308.739  317.40  301.170  301.54  77949082   AAPL
1 2026-06-05  312.860  315.17  307.150  307.34  65310502   AAPL
2 2026-06-04  313.230  313.54  309.650  311.23  44869134   AAPL
3 2026-06-03  314.175  316.94  308.850  310.26  50836705   AAPL
4 2026-06-02  307.460  315.45  306.685  315.20  44534716   AAPL
date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker               str
dtype: object


In [52]:
import pyarrow as pa # per lavorare con i dati in formato Parquet
import pyarrow.parquet as pq # per salvare i dati in formato Parquet

output_path = f"../staging/prices/{ticker}_prices.parquet" # percorso di output per il file Parquet, con il nome del ticker per identificare il titolo

os.makedirs(f"../staging/prices", exist_ok=True) # crea la cartella di destinazione se non esiste già

df.to_parquet(output_path, index=False) # metodo to_parquet() per salvare il DataFrame in formato Parquet, index=False per non salvare l'indice come colonna
print(f"File salvato in: {output_path}") 

df_check = pd.read_parquet(output_path)
print(df_check.head())
print(df_check.dtypes)

File salvato in: ../staging/prices/AAPL_prices.parquet
        date     open    high      low   close    volume ticker
0 2026-06-08  308.739  317.40  301.170  301.54  77949082   AAPL
1 2026-06-05  312.860  315.17  307.150  307.34  65310502   AAPL
2 2026-06-04  313.230  313.54  309.650  311.23  44869134   AAPL
3 2026-06-03  314.175  316.94  308.850  310.26  50836705   AAPL
4 2026-06-02  307.460  315.45  306.685  315.20  44534716   AAPL
date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker               str
dtype: object


Abbiamo completato l'esplorazione dell'endpoint TIME_SERIES_DAILY:  
✅ chiamata API     
✅ struttura JSON capita        
✅ trasformazione in DataFrame       
✅ tipi corretti        
✅ salvataggio in Parquet verificato        